## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_circles, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
np.random.seed(42)

print("Libraries loaded successfully!")

## 6.1 Activation Functions

Activation functions introduce **non-linearity** into neural networks.

**Common Activations**:
1. **Sigmoid**: $\sigma(z) = \frac{1}{1 + e^{-z}}$
2. **Tanh**: $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$
3. **ReLU**: $\text{ReLU}(z) = \max(0, z)$
4. **Leaky ReLU**: $\text{LeakyReLU}(z) = \max(0.01z, z)$

In [ ]:
# Activation functions and derivatives
def sigmoid(z):
    """Sigmoid activation."""
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_derivative(z):
    """Derivative of sigmoid."""
    s = sigmoid(z)
    return s * (1 - s)

def tanh(z):
    """Hyperbolic tangent."""
    return np.tanh(z)

def tanh_derivative(z):
    """Derivative of tanh."""
    return 1 - np.tanh(z)**2

def relu(z):
    """Rectified Linear Unit."""
    return np.maximum(0, z)

def relu_derivative(z):
    """Derivative of ReLU."""
    return (z > 0).astype(float)

def leaky_relu(z, alpha=0.01):
    """Leaky ReLU."""
    return np.where(z > 0, z, alpha * z)

def leaky_relu_derivative(z, alpha=0.01):
    """Derivative of Leaky ReLU."""
    return np.where(z > 0, 1, alpha)

# Visualize activations
z = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sigmoid
axes[0, 0].plot(z, sigmoid(z), 'b-', linewidth=2, label='σ(z)')
axes[0, 0].plot(z, sigmoid_derivative(z), 'r--', linewidth=2, label="σ'(z)")
axes[0, 0].axhline(0, color='black', linewidth=0.5)
axes[0, 0].axvline(0, color='black', linewidth=0.5)
axes[0, 0].set_title('Sigmoid', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('z', fontsize=11)
axes[0, 0].set_ylabel('Activation', fontsize=11)
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Tanh
axes[0, 1].plot(z, tanh(z), 'b-', linewidth=2, label='tanh(z)')
axes[0, 1].plot(z, tanh_derivative(z), 'r--', linewidth=2, label="tanh'(z)")
axes[0, 1].axhline(0, color='black', linewidth=0.5)
axes[0, 1].axvline(0, color='black', linewidth=0.5)
axes[0, 1].set_title('Tanh', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('z', fontsize=11)
axes[0, 1].set_ylabel('Activation', fontsize=11)
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# ReLU
axes[1, 0].plot(z, relu(z), 'b-', linewidth=2, label='ReLU(z)')
axes[1, 0].plot(z, relu_derivative(z), 'r--', linewidth=2, label="ReLU'(z)")
axes[1, 0].axhline(0, color='black', linewidth=0.5)
axes[1, 0].axvline(0, color='black', linewidth=0.5)
axes[1, 0].set_title('ReLU', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('z', fontsize=11)
axes[1, 0].set_ylabel('Activation', fontsize=11)
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Leaky ReLU
axes[1, 1].plot(z, leaky_relu(z), 'b-', linewidth=2, label='Leaky ReLU(z)')
axes[1, 1].plot(z, leaky_relu_derivative(z), 'r--', linewidth=2, label="Leaky ReLU'(z)")
axes[1, 1].axhline(0, color='black', linewidth=0.5)
axes[1, 1].axvline(0, color='black', linewidth=0.5)
axes[1, 1].set_title('Leaky ReLU', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('z', fontsize=11)
axes[1, 1].set_ylabel('Activation', fontsize=11)
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nActivation Properties:")
print("- Sigmoid: Range (0,1), vanishing gradient for |z| > 3")
print("- Tanh: Range (-1,1), zero-centered, still vanishing gradient")
print("- ReLU: No vanishing gradient for z>0, dead neurons for z<0")
print("- Leaky ReLU: Fixes dying ReLU problem")

## 6.2 Simple Neural Network Implementation

Let's build a 2-layer neural network from scratch:
- Input layer: $n$ features
- Hidden layer: $h$ units with ReLU
- Output layer: 1 unit with sigmoid (binary classification)

In [ ]:
class NeuralNetwork:
    """
    Simple 2-layer neural network for binary classification.
    Architecture: Input -> Hidden (ReLU) -> Output (Sigmoid)
    """
    def __init__(self, input_size, hidden_size, learning_rate=0.01):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        
        # Xavier initialization
        self.W1 = np.random.randn(hidden_size, input_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((hidden_size, 1))
        self.W2 = np.random.randn(1, hidden_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, 1))
        
        # Cache for backprop
        self.cache = {}
        self.costs = []
    
    def forward(self, X):
        """
        Forward propagation.
        X: (n_features, m_samples)
        """
        # Layer 1
        Z1 = self.W1 @ X + self.b1  # (hidden_size, m)
        A1 = relu(Z1)
        
        # Layer 2
        Z2 = self.W2 @ A1 + self.b2  # (1, m)
        A2 = sigmoid(Z2)
        
        # Cache for backprop
        self.cache = {
            'X': X,
            'Z1': Z1,
            'A1': A1,
            'Z2': Z2,
            'A2': A2
        }
        
        return A2
    
    def compute_cost(self, Y):
        """
        Compute binary cross-entropy cost.
        Y: (1, m)
        """
        m = Y.shape[1]
        A2 = self.cache['A2']
        
        # Clip to avoid log(0)
        A2_clipped = np.clip(A2, 1e-10, 1 - 1e-10)
        
        cost = -1/m * np.sum(Y * np.log(A2_clipped) + (1 - Y) * np.log(1 - A2_clipped))
        return cost
    
    def backward(self, Y):
        """
        Backpropagation.
        """
        m = Y.shape[1]
        X = self.cache['X']
        A1 = self.cache['A1']
        A2 = self.cache['A2']
        Z1 = self.cache['Z1']
        
        # Layer 2 gradients
        dZ2 = A2 - Y  # (1, m)
        dW2 = 1/m * dZ2 @ A1.T  # (1, hidden_size)
        db2 = 1/m * np.sum(dZ2, axis=1, keepdims=True)  # (1, 1)
        
        # Layer 1 gradients
        dA1 = self.W2.T @ dZ2  # (hidden_size, m)
        dZ1 = dA1 * relu_derivative(Z1)  # (hidden_size, m)
        dW1 = 1/m * dZ1 @ X.T  # (hidden_size, n_features)
        db1 = 1/m * np.sum(dZ1, axis=1, keepdims=True)  # (hidden_size, 1)
        
        return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}
    
    def update_parameters(self, gradients):
        """
        Gradient descent update.
        """
        self.W1 -= self.learning_rate * gradients['dW1']
        self.b1 -= self.learning_rate * gradients['db1']
        self.W2 -= self.learning_rate * gradients['dW2']
        self.b2 -= self.learning_rate * gradients['db2']
    
    def train(self, X, Y, num_iterations=1000, print_cost=False):
        """
        Train the neural network.
        X: (n_features, m)
        Y: (1, m)
        """
        for i in range(num_iterations):
            # Forward
            A2 = self.forward(X)
            
            # Cost
            cost = self.compute_cost(Y)
            self.costs.append(cost)
            
            # Backward
            gradients = self.backward(Y)
            
            # Update
            self.update_parameters(gradients)
            
            if print_cost and i % 100 == 0:
                print(f"Iteration {i}: Cost = {cost:.4f}")
    
    def predict(self, X):
        """
        Predict class labels.
        """
        A2 = self.forward(X)
        predictions = (A2 > 0.5).astype(int)
        return predictions

print("Neural Network class implemented!")

## 6.3 Training on XOR Problem

XOR is not linearly separable → perfect test for neural networks!

In [ ]:
# XOR dataset
X_xor = np.array([[0, 0, 1, 1],
                  [0, 1, 0, 1]])
Y_xor = np.array([[0, 1, 1, 0]])

# Train neural network
nn_xor = NeuralNetwork(input_size=2, hidden_size=4, learning_rate=0.5)
nn_xor.train(X_xor, Y_xor, num_iterations=5000, print_cost=True)

# Predictions
predictions = nn_xor.predict(X_xor)
accuracy = np.mean(predictions == Y_xor)

print(f"\nFinal Accuracy: {accuracy:.2%}")
print("\nPredictions vs True:")
for i in range(4):
    print(f"  Input: {X_xor[:, i]}, Predicted: {predictions[0, i]}, True: {int(Y_xor[0, i])}")

# Plot cost
plt.figure(figsize=(10, 6))
plt.plot(nn_xor.costs, linewidth=2)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Cost', fontsize=12)
plt.title('XOR Problem: Training Cost', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6.4 Decision Boundary Visualization

In [ ]:
# Generate non-linear dataset
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.3, random_state=42)

# Prepare data
X_circles_T = X_circles.T
Y_circles = y_circles.reshape(1, -1)

# Train neural network
nn_circles = NeuralNetwork(input_size=2, hidden_size=8, learning_rate=0.1)
nn_circles.train(X_circles_T, Y_circles, num_iterations=3000)

# Create mesh for decision boundary
h = 0.01
x_min, x_max = X_circles[:, 0].min() - 0.5, X_circles[:, 0].max() + 0.5
y_min, y_max = X_circles[:, 1].min() - 0.5, X_circles[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Predict on mesh
Z = nn_circles.predict(np.c_[xx.ravel(), yy.ravel()].T)
Z = Z.reshape(xx.shape)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Decision boundary
axes[0].contourf(xx, yy, Z, alpha=0.6, cmap='RdBu')
axes[0].scatter(X_circles[y_circles==0, 0], X_circles[y_circles==0, 1],
               c='blue', marker='o', s=50, edgecolors='k', alpha=0.7, label='Class 0')
axes[0].scatter(X_circles[y_circles==1, 0], X_circles[y_circles==1, 1],
               c='red', marker='s', s=50, edgecolors='k', alpha=0.7, label='Class 1')
axes[0].set_xlabel('Feature 1', fontsize=12)
axes[0].set_ylabel('Feature 2', fontsize=12)
axes[0].set_title('Neural Network Decision Boundary', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Cost curve
axes[1].plot(nn_circles.costs, linewidth=2)
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('Cost', fontsize=12)
axes[1].set_title('Training Cost', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Accuracy
predictions = nn_circles.predict(X_circles_T)
accuracy = np.mean(predictions == Y_circles)
print(f"\nAccuracy: {accuracy:.2%}")

## 6.5 MNIST Digit Classification

Let's apply our neural network to handwritten digit recognition.

In [ ]:
# Load digits dataset (1797 samples, 8x8 images)
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

# Binary classification: 0 vs 1
mask = (y_digits == 0) | (y_digits == 1)
X_binary = X_digits[mask]
y_binary = y_digits[mask]

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X_binary, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Prepare for neural network (transpose)
X_train_T = X_train_scaled.T
X_test_T = X_test_scaled.T
Y_train = y_train.reshape(1, -1)
Y_test = y_test.reshape(1, -1)

print(f"Training set: {X_train_T.shape}")
print(f"Test set: {X_test_T.shape}")

# Train neural network
nn_digits = NeuralNetwork(input_size=64, hidden_size=32, learning_rate=0.1)
print("\nTraining Neural Network...")
nn_digits.train(X_train_T, Y_train, num_iterations=2000, print_cost=True)

# Evaluate
train_pred = nn_digits.predict(X_train_T)
test_pred = nn_digits.predict(X_test_T)

train_acc = np.mean(train_pred == Y_train)
test_acc = np.mean(test_pred == Y_test)

print(f"\nTraining Accuracy: {train_acc:.2%}")
print(f"Test Accuracy: {test_acc:.2%}")

# Plot some predictions
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i].reshape(8, 8), cmap='gray')
    pred = int(test_pred[0, i])
    true = int(y_test[i])
    color = 'green' if pred == true else 'red'
    ax.set_title(f'Pred: {pred}, True: {true}', color=color, fontsize=10)
    ax.axis('off')

plt.suptitle('MNIST Digit Predictions (0 vs 1)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6.6 Weight Initialization Study

Proper initialization is critical for training deep networks.

**Strategies**:
1. **Zero**: All zeros → symmetry problem
2. **Random Small**: $W \sim \mathcal{N}(0, 0.01^2)$ → vanishing gradients
3. **Random Large**: $W \sim \mathcal{N}(0, 1)$ → exploding gradients
4. **Xavier**: $W \sim \mathcal{N}(0, \frac{1}{n_{in}})$ → good for sigmoid/tanh
5. **He**: $W \sim \mathcal{N}(0, \frac{2}{n_{in}})$ → good for ReLU

In [ ]:
# Test different initializations
class NeuralNetworkInit(NeuralNetwork):
    def __init__(self, input_size, hidden_size, learning_rate=0.01, init_method='xavier'):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        
        # Different initializations
        if init_method == 'zeros':
            self.W1 = np.zeros((hidden_size, input_size))
            self.W2 = np.zeros((1, hidden_size))
        elif init_method == 'small':
            self.W1 = np.random.randn(hidden_size, input_size) * 0.01
            self.W2 = np.random.randn(1, hidden_size) * 0.01
        elif init_method == 'large':
            self.W1 = np.random.randn(hidden_size, input_size) * 1.0
            self.W2 = np.random.randn(1, hidden_size) * 1.0
        elif init_method == 'xavier':
            self.W1 = np.random.randn(hidden_size, input_size) * np.sqrt(1.0 / input_size)
            self.W2 = np.random.randn(1, hidden_size) * np.sqrt(1.0 / hidden_size)
        elif init_method == 'he':
            self.W1 = np.random.randn(hidden_size, input_size) * np.sqrt(2.0 / input_size)
            self.W2 = np.random.randn(1, hidden_size) * np.sqrt(2.0 / hidden_size)
        
        self.b1 = np.zeros((hidden_size, 1))
        self.b2 = np.zeros((1, 1))
        self.cache = {}
        self.costs = []

# Test on circles dataset
init_methods = ['small', 'large', 'xavier', 'he']
results = {}

for method in init_methods:
    nn = NeuralNetworkInit(input_size=2, hidden_size=8, 
                          learning_rate=0.1, init_method=method)
    nn.train(X_circles_T, Y_circles, num_iterations=1000)
    results[method] = nn.costs

# Plot comparison
plt.figure(figsize=(12, 7))
for method, costs in results.items():
    plt.plot(costs, linewidth=2, label=method.capitalize())

plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Cost', fontsize=12)
plt.title('Weight Initialization Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nFinal Costs:")
for method, costs in results.items():
    print(f"  {method.capitalize()}: {costs[-1]:.4f}")

## 6.7 Gradient Checking

Verify backpropagation implementation using numerical gradients:
$$\frac{\partial J}{\partial \theta} \approx \frac{J(\theta + \epsilon) - J(\theta - \epsilon)}{2\epsilon}$$

In [ ]:
def gradient_check(nn, X, Y, epsilon=1e-7):
    """
    Check gradients using numerical approximation.
    """
    # Forward and backward
    nn.forward(X)
    gradients = nn.backward(Y)
    
    # Get parameters
    params = {'W1': nn.W1, 'b1': nn.b1, 'W2': nn.W2, 'b2': nn.b2}
    
    # Numerical gradients
    num_gradients = {}
    
    for param_name, param in params.items():
        num_grad = np.zeros_like(param)
        
        # Iterate through each element
        it = np.nditer(param, flags=['multi_index'], op_flags=['readwrite'])
        
        count = 0
        while not it.finished and count < 10:  # Check only first 10 for speed
            idx = it.multi_index
            old_value = param[idx]
            
            # J(theta + epsilon)
            param[idx] = old_value + epsilon
            nn.forward(X)
            cost_plus = nn.compute_cost(Y)
            
            # J(theta - epsilon)
            param[idx] = old_value - epsilon
            nn.forward(X)
            cost_minus = nn.compute_cost(Y)
            
            # Numerical gradient
            num_grad[idx] = (cost_plus - cost_minus) / (2 * epsilon)
            
            # Restore
            param[idx] = old_value
            it.iternext()
            count += 1
        
        num_gradients['d' + param_name] = num_grad
    
    # Compare
    print("\nGradient Check Results:")
    for param_name in ['W1', 'b1', 'W2', 'b2']:
        grad_key = 'd' + param_name
        
        # Get non-zero elements for comparison
        mask = num_gradients[grad_key] != 0
        if np.any(mask):
            analytical = gradients[grad_key][mask]
            numerical = num_gradients[grad_key][mask]
            
            diff = np.linalg.norm(analytical - numerical) / \
                   (np.linalg.norm(analytical) + np.linalg.norm(numerical))
            
            status = "✓ PASS" if diff < 1e-5 else "✗ FAIL"
            print(f"  {param_name}: Difference = {diff:.2e} {status}")

# Test gradient checking
nn_check = NeuralNetwork(input_size=2, hidden_size=3, learning_rate=0.01)
X_small = X_xor[:, :2]  # Use small dataset
Y_small = Y_xor[:, :2]

gradient_check(nn_check, X_small, Y_small)

## 6.8 Hidden Layer Visualization

Let's visualize what the hidden layer learns.

In [ ]:
# Train on digits again
nn_vis = NeuralNetwork(input_size=64, hidden_size=16, learning_rate=0.1)
nn_vis.train(X_train_T, Y_train, num_iterations=2000)

# Visualize hidden layer weights
fig, axes = plt.subplots(4, 4, figsize=(10, 10))

for i, ax in enumerate(axes.flat):
    if i < nn_vis.hidden_size:
        # Reshape weights to 8x8 image
        weights = nn_vis.W1[i].reshape(8, 8)
        ax.imshow(weights, cmap='RdBu', aspect='auto')
        ax.set_title(f'Neuron {i+1}', fontsize=9)
    ax.axis('off')

plt.suptitle('Hidden Layer Learned Features\n(Weights from Input to Hidden Layer)',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nEach neuron learns different features of the input digits.")

## Key Takeaways

### 1. **Activation Functions**
- Introduce non-linearity (essential for learning complex patterns)
- **ReLU** is default choice (no vanishing gradient, fast)
- **Sigmoid/Tanh** for output layer (classification probabilities)
- **Leaky ReLU** fixes dying ReLU problem

### 2. **Forward Propagation**
- Compute layer by layer: $z^{[l]} = W^{[l]} a^{[l-1]} + b^{[l]}$, then $a^{[l]} = g(z^{[l]})$
- Cache all intermediate values for backprop

### 3. **Backpropagation**
- Chain rule to compute gradients
- Start from output, propagate backward
- $\delta^{[l]} = (W^{[l+1]})^T \delta^{[l+1]} \odot g'(z^{[l]})$
- Update: $W^{[l]} \leftarrow W^{[l]} - \alpha \frac{\partial J}{\partial W^{[l]}}$

### 4. **Weight Initialization**
- **Never use zeros** (symmetry problem)
- **Xavier** for sigmoid/tanh: $W \sim \mathcal{N}(0, \frac{1}{n_{in}})$
- **He** for ReLU: $W \sim \mathcal{N}(0, \frac{2}{n_{in}})$
- Proper init prevents vanishing/exploding gradients

### 5. **Gradient Checking**
- Use numerical gradients to verify backprop implementation
- Difference should be < $10^{-7}$ (double precision) or < $10^{-5}$ (acceptable)
- **Only for debugging**, turn off during training (slow)

### 6. **Common Pitfalls**
- Vanishing gradients (deep networks, sigmoid/tanh)
- Exploding gradients (poor initialization, high learning rate)
- Dead ReLUs (too many negative z values)
- Poor initialization (symmetry, wrong scale)

### 7. **Best Practices**
- Use ReLU activation (hidden layers)
- He initialization for ReLU networks
- Scale features (StandardScaler)
- Start with small learning rate
- Monitor training cost
- Gradient check for new implementations

### 8. **Architecture Design**
- **Depth**: More layers → more abstract features
- **Width**: More neurons → more capacity
- Start small, increase if underfitting
- Hidden layer size typically between input and output size

## Practice Exercises

1. **Activation Comparison**: Implement a network with different activations (sigmoid, tanh, ReLU). Compare convergence speed and final accuracy.

2. **Deeper Network**: Extend to 3-4 layers. Implement forward and backprop. Does it improve performance on MNIST?

3. **Momentum**: Add momentum to gradient descent: $v_t = \beta v_{t-1} + (1-\beta) \nabla J$. Does training become faster/more stable?

4. **Mini-batch**: Implement mini-batch gradient descent instead of full-batch. Compare speed and convergence.

5. **Batch Normalization**: Add batch norm layer. Verify it helps with vanishing gradients in deep networks.

6. **Vanishing Gradient**: Build 10-layer network with sigmoid. Visualize gradient magnitudes per layer. Confirm vanishing gradient problem.

7. **Dropout**: Implement dropout regularization. Does it improve test accuracy?

8. **Learning Rate Decay**: Implement exponential decay: $\alpha_t = \alpha_0 e^{-kt}$. Plot cost curves.

9. **Multi-class**: Extend to 10-class MNIST (all digits). Use softmax output. Report confusion matrix.

10. **Feature Visualization**: Use t-SNE to visualize hidden layer activations. Do classes cluster?

## References

1. **CS229 Lecture Notes**: Andrew Ng's notes on neural networks
2. **"Gradient-Based Learning Applied to Document Recognition"**: LeCun et al. (1998)
3. **"Understanding the Difficulty of Training Deep Feedforward Neural Networks"**: Glorot & Bengio (2010)
4. **"Delving Deep into Rectifiers"**: He et al. (2015) - He initialization
5. **"Deep Learning"**: Goodfellow et al., Chapters 6-8
6. **"Neural Networks and Deep Learning"**: Michael Nielsen (online book)

---

**Next**: [Lecture 7: Neural Networks - Advanced](07_neural_networks_advanced.ipynb)